In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
spark

In [3]:
import os
import pandas as pd
from datetime import datetime
import re

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [4]:
import nltk
#nltk.download('stopwords')

In [5]:
from pyspark.sql.types import ArrayType, FloatType, NumericType, StringType
from pyspark.ml.feature import Word2Vec, StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

from string import punctuation
from nltk.corpus import stopwords

-------------------------------------------------

In [16]:
total_df = spark.read.parquet("../Data/Textual_data/CEO_CA/trial_ceo_ca.parquet")

In [17]:
total_df.select("channel").distinct().show()

+-------+
|channel|
+-------+
|    aya|
|     cg|
|    cgc|
|    ckg|
|    arg|
+-------+



In [18]:
df_aya = total_df.filter(F.col("ticker") == "AYA.TO")
df_cg = total_df.filter(F.col("ticker") == "CG.TO")
df_cgc = total_df.filter(F.col("ticker") == "CGC.V")
df_ckg = total_df.filter(F.col("ticker") == "CKG.V")
df_arg = total_df.filter(F.col("ticker") == "ARG.TO")

In [19]:
df_aya.write.parquet("../Data/Textual_Data/CEO_CA/AYA.TO_ceo_ca.parquet")
df_cg.write.parquet("../Data/Textual_Data/CEO_CA/CG.TO_ceo_ca.parquet")
df_arg.write.parquet("../Data/Textual_Data/CEO_CA/ARG.TO_ceo_ca.parquet")
df_cgc.write.parquet("../Data/Textual_Data/CEO_CA/CGC.V_ceo_ca.parquet")
df_ckg.write.parquet("../Data/Textual_Data/CEO_CA/CKG.V_ceo_ca.parquet")


---------------------

In [6]:
df = spark.read.parquet("../CEO_CA/DMX.V_ceo_ca.parquet")
df.show()

+-------+--------------------+----------------+------------+--------------------+-----+-----+------+
|channel|           timestamp|        username|    spiel_id|                text|score|votes|ticker|
+-------+--------------------+----------------+------------+--------------------+-----+-----+------+
|    dmx|2024-12-27 01:17:...|        @ronnie2|807a48153b86|@stonecrop..Had a...|  NaN|    0| DMX.V|
|    dmx|2024-12-27 03:20:...|      @metcalfed|58524c15e4f1|Wrong channel @ro...|  1.0|    1| DMX.V|
|    dmx|2024-12-27 05:54:...|        @ronnie2|6027f5c93122|@metcalfed   Good...|  2.0|    2| DMX.V|
|    dmx|2024-12-27 17:53:...|        @onetree|bdd0088d381f|In addition to AH...|  4.0|    5| DMX.V|
|    dmx|2024-12-27 18:15:...|     @LMHradecki|3bbaba303a33|That’s how Garret...|  3.0|    4| DMX.V|
|    dmx|2024-12-28 02:33:...|        @AFChief|d320bb3e3eec|#stockpickingcont...|  2.0|    3| DMX.V|
|    dmx|2024-12-29 22:11:...|     @LMHradecki|261fa7c59d0e|#stockpickingcont...|  5.0|    

In [8]:
# overwrite the Local Date column with ET time instead of CET 
DMX_df = DMX_df.withColumn("Local Date (ET)",
                                F.from_utc_timestamp(
                                    F.to_utc_timestamp(F.col("timestamp"), "CET"), 
                                    "America/New_York")) \
                    .drop("timestamp")

DMX_df.show()

+-------+----------------+------------+--------------------+-----+-----+------+--------------------+
|channel|        username|    spiel_id|                text|score|votes|ticker|     Local Date (ET)|
+-------+----------------+------------+--------------------+-----+-----+------+--------------------+
|    dmx|        @ronnie2|807a48153b86|@stonecrop..Had a...|  NaN|    0| DMX.V|2024-12-26 19:17:...|
|    dmx|      @metcalfed|58524c15e4f1|Wrong channel @ro...|  1.0|    1| DMX.V|2024-12-26 21:20:...|
|    dmx|        @ronnie2|6027f5c93122|@metcalfed   Good...|  2.0|    2| DMX.V|2024-12-26 23:54:...|
|    dmx|        @onetree|bdd0088d381f|In addition to AH...|  4.0|    5| DMX.V|2024-12-27 11:53:...|
|    dmx|     @LMHradecki|3bbaba303a33|That’s how Garret...|  3.0|    4| DMX.V|2024-12-27 12:15:...|
|    dmx|        @AFChief|d320bb3e3eec|#stockpickingcont...|  2.0|    3| DMX.V|2024-12-27 20:33:...|
|    dmx|     @LMHradecki|261fa7c59d0e|#stockpickingcont...|  5.0|    4| DMX.V|2024-12-29 1

In [11]:
DMX_df.show(10, truncate=False)

+-------+-------------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Sentiment

To get a full representation of the sentiment of the comments on CEO.CA for each ticker during each biweekly period we'll apply both Finbert and Vader. We'll include both features that catches finance sentiment (Finbert) and features that catch emotion, hype, excitement, fear (Vader). By combining these we'll sentiment scores will possibly be more powerfull as predictors

## Vader

In [12]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
#nltk.download('vader_lexicon')
import re

In [13]:
sia = SentimentIntensityAnalyzer()

Small preprocessing steps to improve Vader's efficiency. Careful with too much preprocessing, could cause performance drop since Vader is lexicon based.

In [14]:
@F.udf("string")
def preprocess(text):
    if text is None:
        return ""
    
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)

    # Remove @mentions
    text = re.sub(r"@\w+", " ", text)

    # Replace tickers
    text = re.sub(r"\$[A-Za-z\.]+", " TICKER ", text)

    # Replace multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [15]:
DMX_Vader_df = DMX_df.withColumn("clean_text", preprocess("text"))

Creating Sentiment score

In [16]:
@F.udf("double")
def compute_sentiment(text):
    return float(sia.polarity_scores(text)["compound"])

In [17]:
DMX_Vader_df = DMX_Vader_df.withColumn("sentiment", compute_sentiment("clean_text"))

In [18]:
DMX_Vader_df.show()

+-------+----------------+------------+--------------------+-----+-----+------+--------------------+--------------------+---------+
|channel|        username|    spiel_id|                text|score|votes|ticker|     Local Date (ET)|          clean_text|sentiment|
+-------+----------------+------------+--------------------+-----+-----+------+--------------------+--------------------+---------+
|    dmx|        @ronnie2|807a48153b86|@stonecrop..Had a...|  NaN|    0| DMX.V|2024-12-26 19:17:...|..Had a look at t...|   0.9615|
|    dmx|      @metcalfed|58524c15e4f1|Wrong channel @ro...|  1.0|    1| DMX.V|2024-12-26 21:20:...|       Wrong channel|  -0.4767|
|    dmx|        @ronnie2|6027f5c93122|@metcalfed   Good...|  2.0|    2| DMX.V|2024-12-26 23:54:...|Good point! Was a...|   0.7639|
|    dmx|        @onetree|bdd0088d381f|In addition to AH...|  4.0|    5| DMX.V|2024-12-27 11:53:...|In addition to AH...|  -0.3031|
|    dmx|     @LMHradecki|3bbaba303a33|That’s how Garret...|  3.0|    4| DMX

## Roberta

### Preprocess Data for FinBert

In [ ]:
def preprocess_comments(text: str) -> str:
    if text is None:
        return ""
    new_text = []
    for t in str(text).split(" "):
        if t.startswith("@") and len(t) > 1:
            t = "@user"
        elif t.startswith("http"):
            t = "http"
        new_text.append(t)
    return " ".join(new_text)

### Roberta Execution

In [ ]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached https://download.pytorch.org/whl/cu126/torchvision-0.24.1%2Bcu126-cp311-cp311-win_amd64.whl.metadata (6.1 kB)
Using cached https://download.pytorch.org/whl/cu126/torchvision-0.24.1%2Bcu126-cp311-cp311-win_amd64.whl (8.5 MB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#!pip uninstall -y torch torchaudio

#!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126


Found existing installation: torch 2.9.1
Uninstalling torch-2.9.1:
  Successfully uninstalled torch-2.9.1


You can safely remove it manually.


Looking in indexes: https://download.pytorch.org/whl/cu126
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB 4.4 MB/s eta 0:09:54
   ---------------------------------------- 0.0/2.6 GB 9.6 MB/s eta 0:04:30
   ---------------------------------------- 0.0/2.6 GB 21.7 MB/s eta 0:01:59
   ---------------------------------------- 0.0/2.6 GB 31.1 MB/s eta 0:01:23
   ---------------------------------------- 0.0/2.6 GB 37.3 MB/s eta 0:01:10
   ---------------------------------------- 0.0/2.6 GB 54.7 MB/s eta 0:00:48
   ---------------------------------------- 0.0/2.6 GB 59.5 MB/s eta 0:00:44
   ---------------------------------------- 0.0/2.6 GB 59.5 MB/s eta 0:00:44
   ---------------------------------------- 0.0/2.6 GB 59.5 MB/s eta 0:00:44
   ---------------------------------------- 0.0/2.6 GB 59.5 MB/s eta 0:00:44
   ---------------------------------------- 0.0/2.6 GB 59.5 MB/s eta 0:00:44
   ------------------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.9.1+cu126
CUDA available: True
Device count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
#!pip3 install transformers

In [33]:
from pyspark.ml.functions import array_to_vector
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

In [34]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Determine device: GPU is preferred if a CUDA-enabled GPU is available. To install torch with cuda support, follow instructions at https://pytorch.org/get-started/locally/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [35]:
from scipy.special import softmax

def batched_sentiment(texts, batch_size=64):
    """
    texts: list[str]
    returns: list[float]  (sentiment scores: pos - neg)
    """
    scores_all = []

    for i in range(0, len(texts), batch_size):
        batch_raw = texts[i:i+batch_size]
        batch = [preprocess_tweet(t) for t in batch_raw]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**enc)
            logits = outputs.logits  # [batch, 3]

        probs = softmax(logits.cpu().numpy(), axis=1)  # to numpy

        # labels: 0=negative, 1=neutral, 2=positive
        p_neg = probs[:, 0]
        p_pos = probs[:, 2]
        batch_scores = (p_pos - p_neg).tolist()
        scores_all.extend(batch_scores)

    return scores_all

In [36]:
# 1) From Spark to pandas
pdf = DMX_df.select("spiel_id", "text").toPandas()
texts = pdf["text"].fillna("").tolist()

pdf["roberta_score"] = batched_sentiment(texts, batch_size=64)

In [37]:
pdf.head()

,spiel_id,text,roberta_score
0,807a48153b86,@stonecrop..Had a look at the trading of B on ...,0.036380
1,58524c15e4f1,Wrong channel @ronnie2,-0.846130
2,6027f5c93122,@metcalfed Good point! Was asked a question ...,0.892610
3,bdd0088d381f,"In addition to AH Swede's great post above, it...",0.047496
4,3bbaba303a33,That’s how Garrett put it in his latest interv...,-0.087833


In [43]:
DMX_df.show(5)

+-------+-----------+------------+--------------------+-----+-----+------+--------------------+
|channel|   username|    spiel_id|                text|score|votes|ticker|     Local Date (ET)|
+-------+-----------+------------+--------------------+-----+-----+------+--------------------+
|    dmx|   @ronnie2|807a48153b86|@stonecrop..Had a...|  NaN|    0| DMX.V|2024-12-26 19:17:...|
|    dmx| @metcalfed|58524c15e4f1|Wrong channel @ro...|  1.0|    1| DMX.V|2024-12-26 21:20:...|
|    dmx|   @ronnie2|6027f5c93122|@metcalfed   Good...|  2.0|    2| DMX.V|2024-12-26 23:54:...|
|    dmx|   @onetree|bdd0088d381f|In addition to AH...|  4.0|    5| DMX.V|2024-12-27 11:53:...|
|    dmx|@LMHradecki|3bbaba303a33|That’s how Garret...|  3.0|    4| DMX.V|2024-12-27 12:15:...|
+-------+-----------+------------+--------------------+-----+-----+------+--------------------+
only showing top 5 rows


In [49]:
pdf["spiel_id"].value_counts()

spiel_id
47f6ac07c974    50
807a48153b86    50
58524c15e4f1    50
6027f5c93122    50
bdd0088d381f    50
                ..
9bdd228d00d6     5
337d6a09bc74     4
3c7695134016     3
2d92c0886adb     2
19093ab3f522     1
Name: count, Length: 6028, dtype: int64

In [10]:
pdf_small = pdf[["spiel_id", "roberta_score"]]
roberta_sdf = spark.createDataFrame(pdf_small)


In [47]:
pdf.to_csv("../CEO_CA/sent_DMX.csv")

In [6]:
pdf = pd.read_csv("../CEO_CA/sent_DMX.csv")

In [11]:
DMX_with_roberta = (
    DMX_df
    .join(roberta_sdf, on="spiel_id", how="left")
)

DMX_with_roberta.show(5, truncate=False)


+------------+-------+--------+-----------------------------------------------------+-----+-----+------+-----------------------+------------------+
|spiel_id    |channel|username|text                                                 |score|votes|ticker|Local Date (ET)        |roberta_score     |
+------------+-------+--------+-----------------------------------------------------+-----+-----+------+-----------------------+------------------+
|02a73eb00b7e|dmx    |@Yaple  |I’m going to be a multimillionaire from this stock!!!|0.0  |1    |DMX.V |2025-08-27 08:25:33.116|0.9289568662643432|
|02a73eb00b7e|dmx    |@Yaple  |I’m going to be a multimillionaire from this stock!!!|0.0  |1    |DMX.V |2025-08-27 08:25:33.116|0.9289568662643432|
|02a73eb00b7e|dmx    |@Yaple  |I’m going to be a multimillionaire from this stock!!!|0.0  |1    |DMX.V |2025-08-27 08:25:33.116|0.9289568662643432|
|02a73eb00b7e|dmx    |@Yaple  |I’m going to be a multimillionaire from this stock!!!|0.0  |1    |DMX.V |2025-08-

In [13]:
DMX_with_roberta = DMX_with_roberta.dropDuplicates()
DMX_with_roberta.show()

+------------+-------+--------------------+--------------------+-----+-----+------+--------------------+-------------------+
|    spiel_id|channel|            username|                text|score|votes|ticker|     Local Date (ET)|      roberta_score|
+------------+-------+--------------------+--------------------+-----+-----+------+--------------------+-------------------+
|003a3a897d2d|    dmx|          @HHorseman|https://www.etc.s...|  3.0|    3| DMX.V|2025-08-27 11:12:...| 0.0094590187072753|
|00552716532d|    dmx|         @rhinotiger|In that case, on ...|  2.0|    1| DMX.V|2024-09-25 11:01:...|-0.2501480877399444|
|005a1e72f9e2|    dmx|@RandomLossAccumu...|@Auaguco And than...|  1.0|    2| DMX.V|2024-08-31 10:45:...| 0.9350111484527588|
|006d5be24a08|    dmx|      @Glasshalffull|VAP 1.03 break th...|  0.0|    1| DMX.V|2025-08-27 15:26:...| 0.0997276902198791|
|00825910633f|    dmx|          @metcalfed|Lucky you. Pays t...|  1.0|    1| DMX.V|2025-04-07 15:34:...| 0.6518602967262268|


In [18]:
DMX_with_roberta.count()

7466

In [ ]:
#DMX_with_roberta.write.parquet("../CEO_CA/roberta_sentiment/roberta_DMX.parquet")